Alteração territorial provocada pelo SSR em 2025

In [ ]:
from pathlib import Path
import re
import sys
import pandas as pd

sys.path.append("/code/scripts")

from analysis_config import ANALYSIS, seasonal_pairs
from comparison_utils import (
    compare_classes,
    priority_agreement,
    seasonal_effect_summary
)

In [ ]:
pilots = ["fundao", "badajoz"]
all_changes = []
all_agreement = []
matrices = {}

for pilot in pilots:
    inventory = pd.read_csv(
        ANALYSIS / "02_classes" / pilot /
        f"class_inventory_{pilot}.csv"
    )
    paths = dict(zip(
        inventory["map_id"],
        inventory["local_class_raster"]
    ))

    out_dir = ANALYSIS / "07_ssr_effect" / pilot
    out_dir.mkdir(parents=True, exist_ok=True)

    for pair in seasonal_pairs(2025):
        difference = out_dir / f"{pair['comparison_id']}_difference.tif"
        agreement_raster = out_dir / f"{pair['comparison_id']}_priority.tif"

        change, matrix = compare_classes(
            paths[pair["map_a"]],
            paths[pair["map_b"]],
            difference,
            pilot=pilot,
            **pair
        )
        priority = priority_agreement(
            paths[pair["map_a"]],
            paths[pair["map_b"]],
            agreement_raster,
            pilot=pilot,
            **pair
        )

        all_changes.append(change)
        all_agreement.append(priority)
        matrices[f"{pilot}_{pair['comparison_id']}"] = matrix

In [ ]:
changes = pd.concat(all_changes, ignore_index=True)
agreement = pd.concat(all_agreement, ignore_index=True)
summary = seasonal_effect_summary(changes, agreement)

out_dir = ANALYSIS / "07_ssr_effect"
out_dir.mkdir(parents=True, exist_ok=True)
output = out_dir / "ssr_territorial_effect.xlsx"

with pd.ExcelWriter(output) as writer:
    summary.to_excel(
        writer,
        sheet_name="SSR_summary",
        index=False
    )
    changes.to_excel(
        writer,
        sheet_name="Class_changes",
        index=False
    )
    agreement.to_excel(
        writer,
        sheet_name="Priority_agreement",
        index=False
    )

    used_names = set()

    for name, matrix in matrices.items():
        clean = re.sub(r"[^A-Za-z0-9_]", "_", name)[:31]
        sheet = clean
        suffix = 1

        while sheet in used_names:
            suffix += 1
            sheet = f"{clean[:27]}_{suffix}"

        used_names.add(sheet)
        matrix.to_excel(writer, sheet_name=sheet)

print("Resultados guardados em:", output)
summary